# 5. Evaluation — All Models vs RIMS_DRL

Auto-discovers every trained model in `output/` and evaluates it against
RIMS_DRL paper values (Meneghello et al., BPM 2024).

**Framing:** Our agent controls *routing decisions* (which activity next);
RIMS_DRL controls *resource assignment* (which worker does which task).
They use different simulators and time units, so absolute cycle times cannot
be compared. **Relative improvement vs RANDOM** is the only scale-independent
metric comparable across both systems.

## 5.0 Install Dependencies

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
    "pandas", "pyarrow", "numpy", "scipy", "scikit-learn",
    "joblib", "matplotlib", "seaborn", "simpy", "sb3-contrib",
])


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


0

## 5.1 Setup

In [2]:
%matplotlib inline
import sys, json
from pathlib import Path

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "src" / "data_ingestion.py").exists():
        _repo_root = _p
        _src = str(_p / "src")
        if _src not in sys.path: sys.path.insert(0, _src)
        break

OUTPUT_ROOT = _repo_root / "output"
RIMS_DIR    = OUTPUT_ROOT / "RIMS_DRL" / "output"
print(f"Repo root: {_repo_root}")
print(f"RIMS_DIR exists: {RIMS_DIR.exists()}")

Repo root: /mnt/hdd/Code/Git/bwa
RIMS_DIR exists: True


## 5.2 RIMS_DRL Paper Reference Values

From Meneghello et al. (BPM 2024), Table 3 (no calendar, full resources).
All values: mean cycle time in seconds +/- 95% CI, 100 simulation runs.

In [3]:
# Paper values: {policy: (mean_s, ci_half_s)}
RIMS_PAPER = {
    "BPI12W": {
        "label":    "BPI12W (Table 3, no calendar)",
        "n_sims":   100,
        "policies": {
            "RANDOM":     (961,  4),
            "FIFO_Trace": (958,  4),
            "FIFO_Act":   (952,  4),
            "SPT":        (957,  4),
            "DRLHSM":     (900,  4),
        },
    },
    "BPI17W": {
        "label":    "BPI17W (Table 3, no calendar)",
        "n_sims":   25,
        "policies": {
            "RANDOM":     (404, 1),
            "FIFO_Trace": (405, 1),
            "FIFO_Act":   (405, 1),
            "SPT":        (404, 1),
            "DRLHSM":     (404, 1),
        },
    },
}

# Map our dataset names to RIMS paper entries
DATASET_TO_RIMS = {
    "BPIC2012":  "BPI12W",
    "BPIC2012W": "BPI12W",
    "BPIC2017":  "BPI17W",
    "BPIC2015":  None,
}

from evaluation import load_rims_drl_baselines
rims_run = load_rims_drl_baselines(str(RIMS_DIR)) if RIMS_DIR.exists() else {}
if rims_run:
    print("Loaded RIMS_DRL run results:")
    for pol, v in rims_run.items():
        print(f"  {pol:<20} {v['mean']:>8.1f}s  +/-{v['ci_half']:>6.1f}")
else:
    print("No RIMS_DRL run found -- using paper values only")
print()
print("Paper entries available for:", list(RIMS_PAPER.keys()))

Loaded RIMS_DRL run results:
  FIFO_activity          1975.1s  +/-  33.5
  FIFO_case              1912.8s  +/-  26.1
  RANDOM                 2004.2s  +/-  32.9
  SPT                    1967.2s  +/-  37.2
  DRLHSM                 1871.0s  +/-  27.4

Paper entries available for: ['BPI12W', 'BPI17W']


## 5.3 Evaluate All Available Models

Auto-discovers every dataset in `output/` that has a trained model.

In [4]:
import joblib, numpy as np
from collections import Counter
from sb3_contrib import MaskablePPO
from rl_env import ProcessEnv
from kpi_actions import MANAGEMENT_ACTIONS, N_MANAGEMENT_ACTIONS

N_EVAL = 500

def evaluate_dataset(ds_name, n_eps=N_EVAL):
    out = OUTPUT_ROOT / ds_name
    twin_path  = out / f"digital_twin_{ds_name}_train.pkl"
    emb_path   = out / f"activity_embeddings.model"
    model_path = out / "rl_model" / "best_model.zip"
    tc_path    = out / "terminal_classification.json"
    if not twin_path.exists() or not model_path.exists(): return None
    twin     = joblib.load(twin_path)
    embedder = joblib.load(emb_path)
    bad_t = set(json.load(open(tc_path))["bad_terminals"]) if tc_path.exists() else None
    env   = ProcessEnv(twin=twin, embed_model=embedder,
                       kpi_baselines=twin.kpi_baselines, bad_terminals=bad_t)
    model = MaskablePPO.load(model_path, env=env)

    # Action space is MultiDiscrete([max_succ, N_MANAGEMENT_ACTIONS])
    import gymnasium as gym
    _is_multidiscrete = isinstance(env.action_space, gym.spaces.MultiDiscrete)

    def run(use_model):
        rng = np.random.default_rng(42)
        rewards, lengths, terminals, bad_terms = [], [], 0, 0
        mgmt_counts = Counter()
        for _ in range(n_eps):
            obs, _ = env.reset()
            done = truncated = False
            ep_r, steps = 0, 0
            final_act = ''
            while not (done or truncated):
                mask = env.action_masks()
                if use_model:
                    action, _ = model.predict(obs, action_masks=mask, deterministic=True)
                    # action is np.ndarray([routing_idx, mgmt_idx]) for MultiDiscrete
                    if _is_multidiscrete:
                        mgmt_counts[int(action[1])] += 1
                    else:
                        action = int(action)
                else:
                    if _is_multidiscrete:
                        max_succ = env.action_space.nvec[0]
                        routing_mask = mask[:max_succ]
                        mgmt_mask    = mask[max_succ:]
                        routing_idx  = int(rng.choice(np.where(routing_mask)[0]))
                        mgmt_idx     = int(rng.choice(np.where(mgmt_mask)[0]))
                        action = np.array([routing_idx, mgmt_idx])
                        mgmt_counts[mgmt_idx] += 1
                    else:
                        action = int(rng.choice(np.where(mask)[0]))
                obs, r, done, truncated, info = env.step(action)
                ep_r += r; steps += 1
                final_act = info.get('current_activity', '')
            rewards.append(ep_r); lengths.append(steps)
            if done:
                terminals += 1
                if final_act in env._bad_terminals:
                    bad_terms += 1

        # Management action usage rates (fraction of steps)
        total_steps = sum(lengths)
        mgmt_rates = {
            MANAGEMENT_ACTIONS[i].name: mgmt_counts[i] / max(total_steps, 1)
            for i in range(N_MANAGEMENT_ACTIONS)
        }
        return {
            "mean_reward":    float(np.mean(rewards)),
            "std_reward":     float(np.std(rewards)),
            "terminal_rate":  terminals / n_eps,
            "bad_term_rate":  bad_terms / n_eps,
            "mean_length":    float(np.mean(lengths)),
            "mgmt_rates":     mgmt_rates,
        }

    print(f"  {ds_name} ({len(twin.activities)} activities, bad={len(bad_t or [])} terminals)")
    rand_r = run(False)
    rl_r   = run(True)
    rel = (rl_r["mean_reward"] - rand_r["mean_reward"]) / abs(rand_r["mean_reward"]) * 100
    print(f"    RANDOM {rand_r['mean_reward']:+.2f}  RL {rl_r['mean_reward']:+.2f}  improvement {rel:+.1f}%")
    # Show top management actions used by RL
    top_mgmt = sorted(rl_r["mgmt_rates"].items(), key=lambda x: -x[1])[:5]
    print(f"    Top management actions: " + ", ".join(f"{n}={v:.1%}" for n, v in top_mgmt if v > 0))
    return {"random": rand_r, "rl": rl_r, "rel": rel,
            "dataset": ds_name, "n_act": len(twin.activities)}

all_results = {}
print(f"Evaluating all models ({N_EVAL} episodes each)...")
for ds_dir in sorted(OUTPUT_ROOT.iterdir()):
    if not ds_dir.is_dir() or ds_dir.name == "RIMS_DRL": continue
    r = evaluate_dataset(ds_dir.name)
    if r: all_results[ds_dir.name] = r
print(f"Done: {list(all_results.keys())}")


Evaluating all models (500 episodes each)...
  BPIC2012 (24 activities, bad=4 terminals)
    RANDOM -3.79  RL +3.23  improvement +185.2%
    Top management actions: relax_rules_for_low_risk=62.4%, assign_to_primary_team=36.0%, enable_cross_trained_pool=0.6%, merge_tasks_under_role=0.5%, adjust_staffing_by_case_volume=0.4%
Done: ['BPIC2012']


## 5.4 Results Table

In [5]:
import pandas as pd
rows = []
for ds, res in all_results.items():
    rk = DATASET_TO_RIMS.get(ds)
    re = RIMS_PAPER.get(rk, {})
    rims_rel = ""
    if re:
        rr = re["policies"]["RANDOM"][0]
        rd = re["policies"]["DRLHSM"][0]
        rims_rel = f"{(rr-rd)/rr*100:+.1f}%"
    # Top management action used by RL
    mgmt_rates = res["rl"].get("mgmt_rates", {})
    top_mgmt = sorted(mgmt_rates.items(), key=lambda x: -x[1])
    top_mgmt_str = top_mgmt[0][0].replace("_", " ") if top_mgmt else "N/A"
    rows.append({
        "Dataset":              ds,
        "Activities":           res["n_act"],
        "RANDOM reward":        round(res["random"]["mean_reward"], 2),
        "RL reward":            round(res["rl"]["mean_reward"], 2),
        "RL vs RANDOM":         f"{res['rel']:+.1f}%",
        "RL bad term rate":     f"{res['rl']['bad_term_rate']:.1%}",
        "RL ep length":         round(res["rl"]["mean_length"], 1),
        "RND bad term rate":    f"{res['random']['bad_term_rate']:.1%}",
        "Top mgmt action":      top_mgmt_str,
        "RIMS ref":             rk or "N/A",
        "RIMS DRLHSM vs RND":   rims_rel or "N/A",
    })
df_s = pd.DataFrame(rows)
print(df_s.to_string(index=False))
df_s.to_csv(OUTPUT_ROOT / "evaluation_all_models.csv", index=False)
print(f"Saved to {OUTPUT_ROOT / 'evaluation_all_models.csv'}")

# ── Management action usage breakdown per dataset ─────────────────────────────
print()
print("Management action usage (RL policy, fraction of steps):")
from kpi_actions import MANAGEMENT_ACTIONS
for ds, res in all_results.items():
    mgmt_rates = res["rl"].get("mgmt_rates", {})
    if not mgmt_rates:
        continue
    print(f"  [{ds}]")
    for a in MANAGEMENT_ACTIONS:
        rate = mgmt_rates.get(a.name, 0.0)
        if rate > 0.005:  # only show actions used >0.5% of steps
            bar = "█" * int(rate * 100)
            print(f"    [{a.index:2d}] {a.name:<40s} {rate:5.1%}  {bar}")


 Dataset  Activities  RANDOM reward  RL reward RL vs RANDOM RL bad term rate  RL ep length RND bad term rate          Top mgmt action RIMS ref RIMS DRLHSM vs RND
BPIC2012          24          -3.79       3.23      +185.2%             0.0%          19.0             50.0% relax rules for low risk   BPI12W              +6.3%
Saved to /mnt/hdd/Code/Git/bwa/output/evaluation_all_models.csv

Management action usage (RL policy, fraction of steps):
  [BPIC2012]
    [ 0] assign_to_primary_team                   36.0%  ████████████████████████████████████
    [ 3] merge_tasks_under_role                    0.5%  
    [10] enable_cross_trained_pool                 0.6%  
    [11] relax_rules_for_low_risk                 62.4%  ██████████████████████████████████████████████████████████████


## 5.5 Dashboard — All Models vs RIMS_DRL

One row per dataset. Columns:
- **Left**: Our RL vs RANDOM reward (our system metric)
- **Centre (blue title)**: Relative improvement vs RANDOM — comparable across systems
- **Right**: RIMS_DRL absolute CT (reference only) or KPI detail if no RIMS ref

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

plt.rcParams.update({
    'axes.facecolor': '#f8f9fa', 'figure.facecolor': 'white',
    'axes.grid': True, 'grid.color': '#dee2e6', 'grid.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10,
})

C = {
    'rl':          '#2196F3',
    'random':      '#FF9800',
    'rims_drlhsm': '#4CAF50',
    'rims_fifo':   '#A5D6A7',
    'rims_spt':    '#C8E6C9',
    'rims_random': '#E8F5E9',
    'grey':        '#9E9E9E',
}
rpc = {
    'DRLHSM':    C['rims_drlhsm'],
    'FIFO_Trace': C['rims_fifo'],
    'FIFO_Act':   C['rims_fifo'],
    'SPT':        C['rims_spt'],
    'RANDOM':     C['rims_random'],
}

datasets = list(all_results.keys())
n_ds = len(datasets)

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: Per-dataset reward + KPI detail  (one row per dataset, 2 cols)
# ═══════════════════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(n_ds, 2, figsize=(14, 4.5 * n_ds))
fig1.subplots_adjust(hspace=0.55, wspace=0.35)
if n_ds == 1:
    axes1 = [axes1]  # ensure iterable

for row, ds in enumerate(datasets):
    res = all_results[ds]
    rr  = res['random']['mean_reward']
    rlr = res['rl']['mean_reward']
    ax_rew, ax_kpi = axes1[row]

    # Left: reward bar chart
    b = ax_rew.bar([0, 1], [rr, rlr],
                   color=[C['random'], C['rl']], alpha=0.85, width=0.45, zorder=3)
    ax_rew.errorbar([0, 1], [rr, rlr],
                    yerr=[res['random']['std_reward'], res['rl']['std_reward']],
                    fmt='none', color='#333', capsize=6, lw=1.5, zorder=4)
    for bar, val in zip(b, [rr, rlr]):
        offset = abs(max(abs(rr), abs(rlr))) * 0.06
        ax_rew.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + (offset if val >= 0 else -offset*2),
                    f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax_rew.set_xticks([0, 1])
    ax_rew.set_xticklabels(['RANDOM', 'Our RL'], fontsize=10)
    ax_rew.axhline(0, color='#bbb', lw=0.8, ls=':')
    ax_rew.set_ylabel('Mean Episode Reward', fontsize=10)
    ax_rew.set_title(f'[{ds}]  Episode Reward\n'
                     f'{res["n_act"]} activities · ProcessEnv routing env',
                     fontweight='bold', fontsize=11)

    # Right: outcome metrics that actually show improvement
    # Bad terminal rate (lower=better): did the agent avoid decline/cancel?
    # Episode length (lower=better): did the agent route efficiently?
    # NOTE: delay_proxy and rework_norm are NOT shown — they scale with
    # episode length, so a longer-running RL agent always looks worse on
    # those metrics even when it's correctly avoiding bad terminals.
    x2 = np.arange(2)
    bw = 0.32
    rand_vals = [res['random']['bad_term_rate'] * 100, res['random']['mean_length']]
    rl_vals   = [res['rl']['bad_term_rate']     * 100, res['rl']['mean_length']]
    b_r = ax_kpi.bar(x2 - bw/2, rand_vals, bw, label='RANDOM', color=C['random'], alpha=0.85, zorder=3)
    b_l = ax_kpi.bar(x2 + bw/2, rl_vals,   bw, label='Our RL', color=C['rl'],     alpha=0.85, zorder=3)
    ymax = max(rand_vals + rl_vals)
    for xi, (rv, lv) in enumerate(zip(rand_vals, rl_vals)):
        ax_kpi.text(xi - bw/2, rv + ymax*0.02, f'{rv:.1f}', ha='center', va='bottom', fontsize=8)
        ax_kpi.text(xi + bw/2, lv + ymax*0.02, f'{lv:.1f}', ha='center', va='bottom', fontsize=8)
    ax_kpi.set_xticks(x2)
    ax_kpi.set_xticklabels(['Bad terminal rate (%)', 'Episode length (steps)'], fontsize=10)
    ax_kpi.set_ylabel('Value  (lower = better)', fontsize=10)
    ax_kpi.set_title(f'[{ds}]  Outcome Metrics\nBad terminal rate & episode length (lower is better)',
                     fontweight='bold', fontsize=11)
    ax_kpi.legend(fontsize=9)

fig1.suptitle('Our Models — Per-Dataset Reward & Outcome Metrics',
              fontsize=13, fontweight='bold', y=1.01)
fig1.savefig(OUTPUT_ROOT / 'eval_fig1_our_models.png', dpi=150, bbox_inches='tight')
display(fig1)
plt.close(fig1)
print(f'Fig 1 saved.')

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: Relative improvement vs RANDOM — horizontal bars, one panel per dataset
# This is the ONLY metric directly comparable across systems.
# ═══════════════════════════════════════════════════════════════════════════════
fig2, axes2 = plt.subplots(1, n_ds, figsize=(7 * n_ds, 7))
fig2.subplots_adjust(wspace=0.45)
if n_ds == 1:
    axes2 = [axes2]

for col, ds in enumerate(datasets):
    res = all_results[ds]
    rk  = DATASET_TO_RIMS.get(ds)
    re  = RIMS_PAPER.get(rk)
    ax  = axes2[col]

    labels, vals, colors, groups = [], [], [], []

    # Our models
    rr = res['random']['mean_reward']
    labels.append(f'Our RANDOM [{ds}]');  vals.append(0.0);  colors.append(C['random']);  groups.append('ours')
    rel_rl = (res['rl']['mean_reward'] - rr) / abs(rr) * 100
    labels.append(f'Our RL [{ds}]');      vals.append(rel_rl); colors.append(C['rl']);     groups.append('ours')

    # RIMS models
    if re:
        rrc = re['policies']['RANDOM'][0]
        for pol, (ms, _) in re['policies'].items():
            rel = (rrc - ms) / rrc * 100
            labels.append(f'RIMS {pol} [{rk}]')
            vals.append(rel)
            colors.append(rpc.get(pol, C['grey']))
            groups.append('rims')

    # Horizontal bar chart
    y_pos = np.arange(len(labels))
    bars = ax.barh(y_pos, vals, color=colors, alpha=0.88, height=0.6, zorder=3)
    ax.axvline(0, color='#555', lw=1.2, ls='--', zorder=2)

    # Value labels — placed outside bars to avoid overlap
    x_max = max(abs(v) for v in vals) if vals else 1
    for bar, val in zip(bars, vals):
        xpos = val + x_max * 0.03 if val >= 0 else val - x_max * 0.03
        ha   = 'left' if val >= 0 else 'right'
        ax.text(xpos, bar.get_y() + bar.get_height()/2,
                f'{val:+.1f}%', va='center', ha=ha, fontsize=8.5, fontweight='bold')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Improvement vs RANDOM (%)', fontsize=10)
    rims_note = f'vs RIMS [{rk}]' if rk else '(no RIMS ref)'
    ax.set_title(f'[{ds}]  Relative Improvement vs RANDOM\n{rims_note}',
                 fontweight='bold', fontsize=11, color='#1a237e')

    # Divider between our models and RIMS
    if re:
        ax.axhline(1.5, color='#aaa', lw=1.2, ls=':', zorder=2)
        xlim = ax.get_xlim()
        ax.text(xlim[1], 1.5, '  RIMS_DRL', fontsize=8, color='#555', va='bottom')
        ax.text(xlim[1], 1.5, '  Our models', fontsize=8, color='#555', va='top')

    ax.set_xlim(left=min(vals + [0]) * 1.25, right=max(vals + [0]) * 1.35)

legend_handles = [
    mpatches.Patch(color=C['rl'],          label='Our RL'),
    mpatches.Patch(color=C['random'],      label='Our RANDOM'),
    mpatches.Patch(color=C['rims_drlhsm'], label='RIMS DRLHSM'),
    mpatches.Patch(color=C['rims_fifo'],   label='RIMS FIFO'),
    mpatches.Patch(color=C['rims_spt'],    label='RIMS SPT'),
    mpatches.Patch(color=C['rims_random'], label='RIMS RANDOM'),
]
fig2.legend(handles=legend_handles, loc='upper center', ncol=6,
            bbox_to_anchor=(0.5, 1.04), fontsize=9, framealpha=0.9)
fig2.suptitle(
    'Relative Improvement vs RANDOM  —  Scale-independent cross-system comparison\n'
    'This is the ONLY metric directly comparable between Our System and RIMS_DRL.',
    fontsize=12, fontweight='bold', y=1.09, color='#1a237e',
)
fig2.savefig(OUTPUT_ROOT / 'eval_fig2_relative_improvement.png', dpi=150, bbox_inches='tight')
display(fig2)
plt.close(fig2)
print(f'Fig 2 saved.')

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3: RIMS_DRL absolute cycle times (reference only)
# ═══════════════════════════════════════════════════════════════════════════════
rims_datasets = [(ds, DATASET_TO_RIMS.get(ds)) for ds in datasets if DATASET_TO_RIMS.get(ds)]
if rims_datasets:
    fig3, axes3 = plt.subplots(1, len(rims_datasets), figsize=(7 * len(rims_datasets), 5))
    fig3.subplots_adjust(wspace=0.4)
    if len(rims_datasets) == 1:
        axes3 = [axes3]

    for col, (ds, rk) in enumerate(rims_datasets):
        re  = RIMS_PAPER[rk]
        ax  = axes3[col]
        rp  = list(re['policies'].keys())
        rm  = [re['policies'][p][0] for p in rp]
        ri  = [re['policies'][p][1] for p in rp]
        x3  = np.arange(len(rp))
        b3  = ax.bar(x3, rm, color=[rpc.get(p, C['grey']) for p in rp],
                     alpha=0.85, width=0.55, zorder=3)
        ax.errorbar(x3, rm, yerr=ri, fmt='none', color='#333', capsize=5, lw=1.5, zorder=4)
        for bar, val, ci in zip(b3, rm, ri):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(rm) * 0.015,
                    f'{val:.0f}s\n±{ci:.0f}', ha='center', va='bottom', fontsize=8.5)
        ax.set_xticks(x3)
        ax.set_xticklabels([p.replace('_', ' ') for p in rp], fontsize=9)
        ax.set_ylabel('Mean Cycle Time (seconds)', fontsize=10)
        ax.set_title(f'RIMS_DRL [{rk}]\n{re["label"]}',
                     fontweight='bold', fontsize=11)
        ax.set_ylim(0, max(rm) * 1.18)

    fig3.text(0.5, -0.04,
              'These cycle times use LSTM-predicted processing time (minutes scale).\n'
              'Our system uses empirical calendar gaps (hours-days scale). Do NOT compare absolute values.',
              ha='center', fontsize=9, color='#c0392b', style='italic')
    fig3.suptitle('RIMS_DRL Absolute Cycle Times  —  Reference Only\n'
                  '(Different simulator and time units from our system)',
                  fontsize=12, fontweight='bold', y=1.04, color='#555')
    fig3.savefig(OUTPUT_ROOT / 'eval_fig3_rims_ct.png', dpi=150, bbox_inches='tight')
    display(fig3)
    plt.close(fig3)
    print(f'Fig 3 saved.')

print(f'\nAll figures saved to {OUTPUT_ROOT}')

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 4: Management action usage — RL vs RANDOM per dataset
# ═══════════════════════════════════════════════════════════════════════════════
from kpi_actions import MANAGEMENT_ACTIONS, N_MANAGEMENT_ACTIONS

fig4, axes4 = plt.subplots(1, n_ds, figsize=(max(10, 7 * n_ds), 6))
fig4.subplots_adjust(wspace=0.45)
if n_ds == 1:
    axes4 = [axes4]

for col, ds in enumerate(datasets):
    res = all_results[ds]
    ax  = axes4[col]

    rl_rates   = res["rl"].get("mgmt_rates", {})
    rand_rates = res["random"].get("mgmt_rates", {})

    action_names = [a.name.replace("_", "\n") for a in MANAGEMENT_ACTIONS]
    rl_vals   = [rl_rates.get(a.name, 0.0) * 100   for a in MANAGEMENT_ACTIONS]
    rand_vals = [rand_rates.get(a.name, 0.0) * 100 for a in MANAGEMENT_ACTIONS]

    x4 = np.arange(N_MANAGEMENT_ACTIONS)
    bw = 0.38
    ax.bar(x4 - bw/2, rand_vals, bw, label='RANDOM', color=C['random'], alpha=0.75, zorder=3)
    ax.bar(x4 + bw/2, rl_vals,   bw, label='Our RL', color=C['rl'],     alpha=0.85, zorder=3)
    ax.set_xticks(x4)
    ax.set_xticklabels(action_names, fontsize=6, rotation=45, ha='right')
    ax.set_ylabel('Usage Rate (%)', fontsize=9)
    ax.set_title(f'[{ds}]  Management Action Usage\nRL vs RANDOM (% of steps)',
                 fontweight='bold', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylim(0, max(max(rl_vals), max(rand_vals), 1) * 1.25)

fig4.suptitle('Management Action Usage — KPI-Based Interventions\n'
              'Shows which actions the RL policy learned to use vs random baseline',
              fontsize=12, fontweight='bold', y=1.03)
fig4.savefig(OUTPUT_ROOT / 'eval_fig4_mgmt_actions.png', dpi=150, bbox_inches='tight')
display(fig4)
plt.close(fig4)
print(f'Fig 4 saved.')

print(f'\nAll figures saved to {OUTPUT_ROOT}')
